In [1]:
%pip install -U datasets huggingface_hub pyarrow

Note: you may need to restart the kernel to use updated packages.


**MMLongBench-Doc**

In [2]:
from pathlib import Path

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "pyproject.toml").exists()
)

EXTERNAL = ROOT / "data" / "external"

for name in [
    "mmlongbench_doc",
    "vidore",
    "omnidocbench",
    "doclaynet",
]:
    (EXTERNAL / name).mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Dataset directory:", EXTERNAL)

Project root: /Users/macbookair/Documents/ENSAE/side project/Multimodal-Document-Intelligence
Dataset directory: /Users/macbookair/Documents/ENSAE/side project/Multimodal-Document-Intelligence/data/external


In [3]:
from huggingface_hub import hf_hub_download
from datasets import load_dataset

dataset_id = "yubo2333/MMLongBench-Doc"
local_dir = EXTERNAL / "mmlongbench_doc"

metadata_path = hf_hub_download(
    repo_id=dataset_id,
    filename="data/train-00000-of-00001.parquet",
    repo_type="dataset",
    local_dir=str(local_dir),
)

qa = load_dataset(
    "parquet",
    data_files=str(metadata_path),
    split="train",
)

print("Questions:", len(qa))
print("Columns:", qa.column_names)

print(qa[0])

Generating train split: 0 examples [00:00, ? examples/s]

Questions: 1091
Columns: ['doc_id', 'doc_type', 'question', 'answer', 'evidence_pages', 'evidence_sources', 'answer_format']
{'doc_id': 'PH_2016.06.08_Economy-Final.pdf', 'doc_type': 'Research report / Introduction', 'question': 'According to the report, how do 5% of the Latinos see economic upward mobility for their children?', 'answer': 'Less well-off', 'evidence_pages': '[5]', 'evidence_sources': "['Chart']", 'answer_format': 'Str'}


In [4]:
sample = qa[0]
pdf_name = sample["doc_id"]

pdf_path = hf_hub_download(
    repo_id=dataset_id,
    filename=f"documents/{pdf_name}",
    repo_type="dataset",
    local_dir=str(local_dir),
)

print("PDF:", pdf_path)
print("Question:", sample["question"])
print("Reference answer:", sample["answer"])
print("Evidence pages:", sample["evidence_pages"])

PDF: /Users/macbookair/Documents/ENSAE/side project/Multimodal-Document-Intelligence/data/external/mmlongbench_doc/documents/PH_2016.06.08_Economy-Final.pdf
Question: According to the report, how do 5% of the Latinos see economic upward mobility for their children?
Reference answer: Less well-off
Evidence pages: [5]


In [5]:
import pymupdf

with pymupdf.open(pdf_path) as doc:
    print("Number of pages:", len(doc))
    print("First page text:")
    print(doc[0].get_text()[:500])

Number of pages: 23
First page text:
 
 
 
FOR RELEASE JUNE 8, 2016 
 
BY Mark Hugo Lopez, Rich Morin and Jens Manuel Krogstad  
 
FOR MEDIA OR OTHER INQUIRIES: 
Mark Hugo Lopez, Director of Hispanic Research 
Brian Mahl, Communications Assistant 
202.419.4372 
www.pewresearch.org 
 
RECOMMENDED CITATION: Lopez, Mark Hugo, Rich Morin and Jens Manuel Krogstad. 2016. “Latinos Increasingly Confident in 
Personal Finances, See Better Economic Times Ahead.” Washington, D.C.: Pew Research Center, June. 
NUMBERS, FACTS AND TRENDS SHAPING 


**Add ViDoRe for retrieval evaluation**

In [6]:
from datasets import load_dataset

vidore_id = "vidore/vidore_v3_finance_fr"
cache = EXTERNAL / "vidore" / "cache"

queries = load_dataset(
    vidore_id,
    data_dir="queries",
    split="test",
    cache_dir=str(cache),
)

qrels = load_dataset(
    vidore_id,
    data_dir="qrels",
    split="test",
    cache_dir=str(cache),
)

print(queries[0])
print(qrels[0])

README.md:   0%|          | 0.00/9.32k [00:00<?, ?B/s]

queries/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  496kB            

queries/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split: 0 examples [00:00, ? examples/s]

qrels/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  129kB            

qrels/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split: 0 examples [00:00, ? examples/s]

{'query_id': 0, 'query': 'Quel était le montant total des rémunérations attribuées à tous les dirigeants de Christian Dior en 2023 ?', 'language': 'french', 'query_types': ['numerical'], 'query_format': 'question', 'content_type': ['Table', 'Text'], 'raw_answers': ['Le montant total des rémunérations attribuées à tous les dirigeants de Christian Dior en 2023 était de 25 millions d’euros.'], 'query_generator': 'sdg', 'query_generation_pipeline': 'custom_pipeline', 'source_type': 'summary', 'query_type_for_generation': 'numerical', 'answer': 'Le montant total de la rémunération attribuée à tous les cadres de Christian Dior en 2023 était de 10 684 415 euros.'}
{'query_id': 0, 'corpus_id': 579, 'score': 1, 'content_type': ['Table', 'Table', 'Text'], 'bounding_boxes': [{'annotator': 0, 'x1': 154, 'x2': 1586, 'y1': 688, 'y2': 1315}, {'annotator': 0, 'x1': 157, 'x2': 1577, 'y1': 1449, 'y2': 1545}, {'annotator': 0, 'x1': 154, 'x2': 1580, 'y1': 1611, 'y2': 1676}]}


In [7]:
from datasets import load_dataset

vidore_id = "vidore/vidore_v3_finance_fr"
cache = EXTERNAL / "vidore" / "cache"

queries = load_dataset(
    vidore_id,
    data_dir="queries",
    split="test",
    cache_dir=str(cache),
)

qrels = load_dataset(
    vidore_id,
    data_dir="qrels",
    split="test",
    cache_dir=str(cache),
)

print(queries[0])
print(qrels[0])

{'query_id': 0, 'query': 'Quel était le montant total des rémunérations attribuées à tous les dirigeants de Christian Dior en 2023 ?', 'language': 'french', 'query_types': ['numerical'], 'query_format': 'question', 'content_type': ['Table', 'Text'], 'raw_answers': ['Le montant total des rémunérations attribuées à tous les dirigeants de Christian Dior en 2023 était de 25 millions d’euros.'], 'query_generator': 'sdg', 'query_generation_pipeline': 'custom_pipeline', 'source_type': 'summary', 'query_type_for_generation': 'numerical', 'answer': 'Le montant total de la rémunération attribuée à tous les cadres de Christian Dior en 2023 était de 10 684 415 euros.'}
{'query_id': 0, 'corpus_id': 579, 'score': 1, 'content_type': ['Table', 'Table', 'Text'], 'bounding_boxes': [{'annotator': 0, 'x1': 154, 'x2': 1586, 'y1': 688, 'y2': 1315}, {'annotator': 0, 'x1': 157, 'x2': 1577, 'y1': 1449, 'y2': 1545}, {'annotator': 0, 'x1': 154, 'x2': 1580, 'y1': 1611, 'y2': 1676}]}


**OmniDocBench for parsing evaluation**

In [8]:
import json

annotations_path = hf_hub_download(
    repo_id="opendatalab/OmniDocBench",
    filename="OmniDocBench.json",
    repo_type="dataset",
    local_dir=str(EXTERNAL / "omnidocbench"),
)

with open(annotations_path, encoding="utf-8") as f:
    annotations = json.load(f)

print("Annotated pages:", len(annotations))
print("First entry:", annotations[0].keys())

OmniDocBench.json: reconstructing file:   0%|          |  0.00B / 42.2MB            

OmniDocBench.json: downloading bytes:           |  0.00B            

Annotated pages: 1651
First entry: dict_keys(['layout_dets', 'page_info', 'extra'])
